# LS-NEQ 00_setup_and_anchor — Setup, Anker und Stimuli

**Dossier-Stufe A** (`lsne.experiments.STAGES["A"]`, siehe `docs/DESIGN.md` §8).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Installiert das Paket `lsne`, bindet Drive ein, prüft die Identität von State 2143 (Sequenz 5 025 183 per HTTP-Range-Request, SHA256 der 2049 Tokens und der 208 Kontext-Tokens, Ziel 22905 „bey“) und baut die Stimulusbank: Familien × Formulierung × Wechselzahl K × Strukturstufe C, mit Konsistenzprüfungen (Ziel-ID, Tail-IDs, Ziel-Leak) und Token-Annotation.

**Eingaben.** `data/stimuli/families/*.json`, `data/stimuli/original_2143.json`, der Anker (Netzwerk, gecacht unter `LSNE_CACHE`).

**Ausgaben.** Tabellen `families`, `variants`, `checks`, `exclusions`, `token_classes`, `tokens`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/A/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/00_setup_and_anchor/`; am Ende eine kurze `summary_00_setup_and_anchor_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Modell → Anker-Verifikation (Hashes, dekodierter Kontext) → Stufe A → Variantentabelle, Ausschlüsse, Tokenklassen-Zählung → Zusammenfassung.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
import os

try:
    from google.colab import drive            # type: ignore
    drive.mount("/content/drive")
except Exception as e:                        # not on Colab, or the mount was refused
    print("Kein Colab-Drive:", repr(e).split("(")[0], "-> lokale Ausgabe")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "00_setup_and_anchor"
STAGE = "A"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(status="reused", run_dir=str(rd.path), seconds=0.0, error="")
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(status="ok", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="")
        print(f"Stufe {stage} ok in {report[stage]['seconds']:.0f} s: {', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Anker-Verifikation: State 2143 ----------------------------------------------------
# The sequence is fetched by HTTP range request (4098 bytes of one shard of
# EleutherAI/pile-standard-pythia-preshuffled), cached under LSNE_CACHE/state2143 and
# compared with the dossier hashes.  The text is never written into the repository.
with guarded("State-2143-Anker"):
    from lsne.anchor import load_anchor, tokenizer_roundtrip_check
    from lsne.config import STATE2143

    anchor = load_anchor(cache_dir=C.cache_dir("state2143"), tokenizer=mb.tokenizer, strict=False)
    print("Sequenzindex      :", STATE2143["sequence_index"], "=", STATE2143["batch_index0"], "*",
          STATE2143["batch_size"], "+", STATE2143["position_in_batch"])
    print("SHA256 Sequenz    :", anchor.sha256_sequence)
    print("   erwartet       :", STATE2143["sha256_sequence"], "| ok:", anchor.sha256_sequence == STATE2143["sha256_sequence"])
    print("SHA256 Kontext    :", anchor.sha256_context)
    print("   erwartet       :", STATE2143["sha256_context"], "| ok:", anchor.sha256_context == STATE2143["sha256_context"],
          "| Dossier-Präfix:", anchor.sha256_context.startswith(STATE2143["sha256_context_prefix_dossier"]))
    print("Ziel-ID           :", anchor.target_id, "| erwartet:", STATE2143["target_id"], "| Zieltext:", repr(anchor.target_text))
    print("Anker verifiziert :", anchor.verified)
    print("Tokenizer-Roundtrip:", tokenizer_roundtrip_check(mb.tokenizer, anchor))
    print("\n--- Dekodierter Kontext (208 Tokens) ---")
    print(anchor.text)
    print("\n--- Fortsetzung im Datensatz (32 Tokens, beginnt mit dem Ziel) ---")
    print(anchor.continuation_text)


In [ ]:
# --- Stufe A ausführen ----------------------------------------------------------------
# CLI-Äquivalent (siehe CLI_COMMAND oben):
#   python -m lsne.cli run --stage A --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT>
res_A = run_stage("A")
print({k: len(v) for k, v in res_A.items()})


In [ ]:
# --- Familien und Varianten -----------------------------------------------------------
with guarded("Familien"):
    fam = table("A", "families")
    show(fam[[c for c in ["tag", "kind", "title", "n_units", "target", "target_id", "dev"] if c in fam.columns]], n=60)

with guarded("Variantentabelle"):
    var = table("A", "variants")
    cols = [c for c in ["tag", "kind", "K", "C", "mask", "n_tokens", "n_words", "n_tokens_D", "F100", "n_switches",
                        "last_switch_pos", "tail_start", "target", "target_id"] if c in var.columns]
    print(f"{len(var)} Varianten; davon original: {int((var['kind'] == 'original').sum()) if 'kind' in var else 0}")
    show(var[cols], n=60)
    std = var[var["kind"] == "standard"] if "kind" in var else var
    print("\nVarianten pro (K, C):")
    show(std.groupby(["K", "C"]).size().rename("n").reset_index())

with guarded("Prüfungen und Ausschlüsse"):
    chk, exc = table("A", "checks"), table("A", "exclusions")
    print(f"{len(chk)} geprüfte Varianten, {len(exc)} ausgeschlossen")
    if len(exc):
        show(exc, n=100)
    else:
        print("Keine Ausschlüsse (Ziel-ID, Tail-IDs und Ziel-Leak in allen Varianten konsistent).")


In [ ]:
# --- Tokenklassen: Anteile pro K und Sprache (Längen-Konfundierung sichtbar machen) -----
with guarded("Tokenklassen"):
    tc = table("A", "token_classes")
    std = tc[tc["kind"] == "standard"] if "kind" in tc else tc
    piv = std.groupby(["K", "klass"])["n"].sum().unstack("klass").fillna(0)
    share = piv.div(piv.sum(axis=1), axis=0)
    ax = share.plot(kind="bar", stacked=True, figsize=(8, 4), width=0.75)
    ax.set_ylabel("Anteil der Tokens")
    ax.set_xlabel("K (Wechselzahl)")
    ax.set_title("Tokenklassen pro K (ws, punct, boundary, word_start, word_cont)")
    ax.legend(title="Klasse", bbox_to_anchor=(1.02, 1), loc="upper left")
    savefig("token_classes_by_K")
    plt.show()
    print("Absolute Zählungen (Klasse × Sprache):")
    show(std.groupby(["lang", "klass"])["n"].sum().unstack("klass").fillna(0).astype(int))

with guarded("Tokens pro Wort nach Sprache"):
    var = table("A", "variants")
    std = var[var["kind"] == "standard"] if "kind" in var else var
    if {"n_tokens_E", "n_words_E", "n_tokens_D", "n_words_D"} <= set(std.columns):
        fig, ax = plt.subplots(figsize=(6, 3.5))
        g = std.groupby("K")[["n_tokens", "n_tokens_D", "n_words_D"]].mean()
        ax.plot(g.index, g["n_tokens"], "o-", label="Tokens gesamt")
        ax.plot(g.index, g["n_tokens_D"], "s-", label="Tokens deutsch")
        ax.plot(g.index, g["n_words_D"], "^-", label="Wörter deutsch")
        ax.set_xlabel("K")
        ax.set_ylabel("Mittel pro Variante")
        ax.set_title("Deutsche Segmente erzeugen mehr Subwort-Tokens (Kovariate, DESIGN §2.4)")
        ax.legend()
        savefig("length_covariate")
        plt.show()


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
